# Fine-Tuning Llama 3.1 8B on Apple Silicon with MLX-LM

## What we're building

A SaaS company gets **50,000 support emails per day**. The CTO wants each email automatically converted into a structured JSON for their CRM:


**The hypothesis:** A small fine-tuned Llama 3.1 8B will beat the large Llama 3.3 70B on this specific task, at 5–10× lower inference cost.


---
## ⚙️ Setup: kernel check + system + dependencies

**Run this one cell first.** It verifies the correct Python environment, checks your Mac, and confirms `mlx-lm` is installed.

In [1]:
import platform
import subprocess
import sys
import importlib.util
from pathlib import Path

# ── SYSTEM INFO ─────────────────────────────────────────────────────
print(f"Python:       {sys.executable}")
print(f"Version:      {sys.version.split()[0]}")
print(f"Architecture: {platform.machine()}")

result = subprocess.run(["sysctl", "hw.memsize"], capture_output=True, text=True)
if result.returncode == 0:
    mem_gb = int(result.stdout.split(": ")[1].strip()) / (1024**3)
    print(f"RAM:          {mem_gb:.0f} GB  {'✅' if mem_gb >= 16 else '⚠️  (<16 GB: use BATCH_SIZE=1 and LORA_LAYERS=4)'}")

print(f"Apple Silicon: {'✅' if platform.machine() == 'arm64' else '❌ not arm64'}")

# ── DEPENDENCIES ─────────────────────────────────────────────────────
PACKAGES = {"mlx_lm": "mlx-lm", "huggingface_hub": "huggingface_hub"}
missing = [pip for imp, pip in PACKAGES.items() if importlib.util.find_spec(imp) is None]

if missing:
    print(f"\nInstalling missing packages: {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing,
                           "--break-system-packages", "-q"])

import mlx_lm
import mlx.core as mx

print(f"\n✅ mlx-lm:    {getattr(mlx_lm, '__version__', 'installed')}")
print(f"✅ Metal GPU: {mx.metal.is_available()}")

if not mx.metal.is_available():
    raise RuntimeError("Metal GPU not available — are you on Apple Silicon?")

Python:       /opt/homebrew/opt/python@3.14/bin/python3.14
Version:      3.14.6
Architecture: arm64
RAM:          16 GB  ✅
Apple Silicon: ✅


/opt/homebrew/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



✅ mlx-lm:    0.31.3
✅ Metal GPU: True


*(Dependencies are installed in the Setup cell above — skip this section.)*

In [2]:
# Moved to Setup cell above — nothing to run here
print("✅ mlx-lm already checked in Setup cell. Continue to HuggingFace login below.")

✅ mlx-lm already checked in Setup cell. Continue to HuggingFace login below.


## 🔑 HuggingFace Authentication

In [ ]:
from huggingface_hub import login, whoami
import os


HF_TOKEN = "" 
# ──────────────────────────────────────────────────────────────────
token = HF_TOKEN or os.environ.get("HF_TOKEN", "")

if not token:
    raise ValueError("Paste your HuggingFace token into HF_TOKEN = \"...\" above, then re-run.")

login(token=token, add_to_git_credential=False)
print(f"✅ Logged in as: {whoami()['name']}")

✅ Logged in as: Bohuslavska


## 📁 Set up file paths


In [4]:
from pathlib import Path
import os
import json

# Auto-detect homework directory (works in Cursor regardless of workspace root)
NOTEBOOK_DIR = Path.cwd()
CANDIDATES = [
    NOTEBOOK_DIR,
    NOTEBOOK_DIR / "homework",
    NOTEBOOK_DIR.parent,
    Path.home() / "Desktop" / "Data_Science" / "ai-engineering" / "lesson 17 - llm-fine-tuning-in-production" / "homework",
]

HOMEWORK_DIR = None
for candidate in CANDIDATES:
    if (candidate / "data" / "eval.jsonl").exists():
        HOMEWORK_DIR = candidate
        break

if HOMEWORK_DIR is None:
    HOMEWORK_DIR = NOTEBOOK_DIR
    print("⚠️  Could not find data/eval.jsonl. Edit HOMEWORK_DIR manually if needed.")

# Define all paths
EVAL_FILE    = HOMEWORK_DIR / "data" / "eval.jsonl"
TRAIN_FILE   = HOMEWORK_DIR / "data" / "train.jsonl"
RESULTS_DIR  = HOMEWORK_DIR / "results"
MLX_DATA_DIR = HOMEWORK_DIR / "mlx_data"   # where MLX-LM will read training data
ADAPTERS_DIR = HOMEWORK_DIR / "adapters"   # where fine-tuned adapter weights are saved

# Create output directories
RESULTS_DIR.mkdir(exist_ok=True)
MLX_DATA_DIR.mkdir(exist_ok=True)
ADAPTERS_DIR.mkdir(exist_ok=True)

# Verify
print(f"Homework dir : {HOMEWORK_DIR}")
print(f"Eval file    : {EVAL_FILE}   {'✅' if EVAL_FILE.exists() else '❌ NOT FOUND'}")
print(f"Train file   : {TRAIN_FILE}  {'✅' if TRAIN_FILE.exists() else '❌ NOT FOUND'}")
print(f"Results dir  : {RESULTS_DIR}")
print(f"Adapters dir : {ADAPTERS_DIR}")

Homework dir : /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/lesson 17 - llm-fine-tuning-in-production/homework
Eval file    : /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/lesson 17 - llm-fine-tuning-in-production/homework/data/eval.jsonl   ✅
Train file   : /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/lesson 17 - llm-fine-tuning-in-production/homework/data/train.jsonl  ✅
Results dir  : /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/lesson 17 - llm-fine-tuning-in-production/homework/results
Adapters dir : /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/lesson 17 - llm-fine-tuning-in-production/homework/adapters


---
## 📊 Step 0: Explore the data

- **`eval.jsonl`** — 30 examples with edge cases (anonymous senders, sarcasm, multi-issue emails). These are our "test" examples. We never train on them.
- **`train.jsonl`** — 300 examples in OpenAI chat format. These are used only for training.


In [5]:
# Load and inspect eval examples
eval_examples = []
with open(EVAL_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            eval_examples.append(json.loads(line))

print(f"Eval set: {len(eval_examples)} examples")
print("\n" + "="*70)
print("SAMPLE EVAL EXAMPLES (what the model must parse correctly)")
print("="*70)

for i, ex in enumerate(eval_examples[:5]):
    print(f"\n[{i+1}] EMAIL:")
    print(f"   {ex['email']}")
    print(f"   EXPECTED JSON: {json.dumps(ex['expected'])}")

Eval set: 30 examples

SAMPLE EVAL EXAMPLES (what the model must parse correctly)

[1] EMAIL:
   Hi, can you tell me when the next maintenance window for Pro Plan is? — A.
   EXPECTED JSON: {"customer_name": null, "product": "Pro Plan", "issue_category": "other", "urgency": "low", "summary": "Maintenance window schedule inquiry"}

[2] EMAIL:
   Hello, two things: (1) my Pro Plan was double charged this month, (2) also dark mode would be nice. Refund is priority. — Maria Bondarenko
   EXPECTED JSON: {"customer_name": "Maria Bondarenko", "product": "Pro Plan", "issue_category": "billing", "urgency": "high", "summary": "Duplicate Pro Plan charge with secondary dark mode feature request"}

[3] EMAIL:
   Production is down. API Access returns 500 on every call. Customers complaining. — DevOps team @ Acme
   EXPECTED JSON: {"customer_name": null, "product": "API Access", "issue_category": "technical", "urgency": "critical", "summary": "Production down, API returning 500"}

[4] EMAIL:
   Oh g

In [6]:
# Show distribution of categories and urgency levels in eval set
from collections import Counter

categories = Counter(ex['expected']['issue_category'] for ex in eval_examples)
urgencies  = Counter(ex['expected']['urgency']        for ex in eval_examples)

print("Eval set distribution:")
print(f"\nCategories:")
for cat, count in sorted(categories.items()):
    bar = "█" * count
    print(f"  {cat:<20} {count:2d}  {bar}")

print(f"\nUrgency levels:")
for urg, count in [("critical",0),("high",0),("medium",0),("low",0)]:
    count = urgencies.get(urg, 0)
    bar = "█" * count
    print(f"  {urg:<20} {count:2d}  {bar}")

print(f"\nNone/anonymous customer_name: {sum(1 for ex in eval_examples if ex['expected']['customer_name'] is None)}")
print("\nNote: urgency accuracy was ~60% for Llama 3.3 70B — that's the main target for improvement.")

Eval set distribution:

Categories:
  account               6  ██████
  billing               7  ███████
  feature_request       5  █████
  other                 5  █████
  technical             7  ███████

Urgency levels:
  critical              3  ███
  high                  6  ██████
  medium               11  ███████████
  low                  10  ██████████

None/anonymous customer_name: 3

Note: urgency accuracy was ~60% for Llama 3.3 70B — that's the main target for improvement.


---
## 🔬 Step 1: Define evaluation helpers

**Metrics we track:**
- `json_valid` — did the model return parseable JSON? (basic sanity check)
- `exact_match` — ALL 5 fields correct simultaneously
- `field_accuracy` — per-field accuracy (shows where the model struggles)

**Matching rules:**
- `customer_name` — relaxed: "John" matches "John Smith" (first name substring)
- `summary` — relaxed: 40% word overlap (free text, hard to be exact)
- `issue_category`, `urgency`, `product` — exact string match

In [7]:
def safe_json_parse(text):
    """
    Parse JSON from model output.
    Handles: markdown fences (```json...```), leading/trailing text,
    special tokens like <|eot_id|> that Llama appends.
    Returns: (parsed_dict_or_None, is_valid_bool)
    """
    text = (text or "").strip()
    # Strip Llama 3.1 special tokens
    for tok in ["<|eot_id|>", "<|end_of_text|>", "<|im_end|>", "<|endoftext|>"]:
        text = text.replace(tok, "")
    text = text.strip()
    # Handle ```json ... ``` code fences
    if text.startswith("```"):
        lines = text.split("\n", 1)
        text = lines[1] if len(lines) > 1 else text
        text = text.rsplit("```", 1)[0].strip()
        if text.startswith("json"):
            text = text[4:].strip()
    # Find the first complete { ... } block
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end > start:
        text = text[start:end+1]
    try:
        return json.loads(text), True
    except json.JSONDecodeError:
        return None, False


def field_match(predicted, expected, field):
    """
    Check if one predicted field matches the expected value.
    Uses relaxed matching for summary (word overlap) and customer_name (first-name substring).
    """
    if field == "summary":
        if not isinstance(predicted, str) or not isinstance(expected, str):
            return False
        # Keep words longer than 3 chars (ignore filler words)
        p = {w.lower().strip(".,!?") for w in predicted.split() if len(w) > 3}
        e = {w.lower().strip(".,!?") for w in expected.split() if len(w) > 3}
        if not p or not e:
            return False
        return len(p & e) / max(len(e), 1) >= 0.4

    if field == "customer_name":
        if predicted is None and expected is None:
            return True
        if predicted is None or expected is None:
            return False
        # First name match: "John" in "John Smith" or "John Doe"
        first_name = str(expected).split()[0].lower()
        return first_name in str(predicted).lower()

    # Exact match for category, urgency, product
    if predicted is None:
        return False
    return str(predicted).strip().lower() == str(expected).strip().lower()


FIELDS = ["customer_name", "product", "issue_category", "urgency", "summary"]

SYSTEM_PROMPT = (
    "You extract structured data from customer support emails. "
    "Return only a single valid JSON object with fields: "
    "customer_name (string or null), product (string), "
    "issue_category (one of: billing, technical, account, feature_request, other), "
    "urgency (one of: low, medium, high, critical), "
    "summary (one short sentence). No extra text."
)

# The 4-bit quantized model from MLX community (same weights as Meta's original, just compressed)
MODEL_ID = "mlx-community/Meta-Llama-3.1-8B-Instruct-4bit"

print("✅ Evaluation helpers defined")
print(f"System prompt: {SYSTEM_PROMPT[:80]}...")
print(f"Model: {MODEL_ID}")

✅ Evaluation helpers defined
System prompt: You extract structured data from customer support emails. Return only a single v...
Model: mlx-community/Meta-Llama-3.1-8B-Instruct-4bit


In [8]:
def evaluate_model(model, tokenizer, examples, label="Model"):
    """
    Run the model on all eval examples and compute metrics.
    
    For each email:
    1. Format as a chat message (system + user)
    2. Generate model response
    3. Parse the JSON from the response
    4. Compare each field against the expected answer
    
    Returns a metrics dict.
    """
    from mlx_lm import generate
    from collections import defaultdict

    results = []
    field_correct = defaultdict(int)
    json_valid = exact_match = 0

    print(f"\n{'='*65}")
    print(f"Evaluating: {label}")
    print(f"{'='*65}")
    print(f"{'#':>3}  {'J':>1}  {'E':>1}  {'cat':>3}  {'urg':>3}  Preview")
    print(f"{'-'*65}")

    for i, ex in enumerate(examples, 1):
        email    = ex["email"]
        expected = ex["expected"]

        # Build the chat prompt (system message + user email)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": email},
        ]

        # apply_chat_template formats the messages using Llama's special tokens:
        # <|begin_of_text|><|start_header_id|>system<|end_header_id|>...<|eot_id|>...
        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,  # adds the assistant turn header so model continues from there
            tokenize=False               # return a string, not token IDs
        )

        # Generate (max 300 tokens is more than enough for our small JSON)
        raw = generate(model, tokenizer, prompt=prompt, max_tokens=300, verbose=False)

        # Parse JSON from the raw output
        predicted, valid = safe_json_parse(raw)
        if valid:
            json_valid += 1

        # Check each field
        per_field = {}
        all_match = True
        for f in FIELDS:
            pred_val = predicted.get(f) if predicted else None
            ok = valid and field_match(pred_val, expected[f], f)
            per_field[f] = ok
            if ok:
                field_correct[f] += 1
            if not ok:
                all_match = False

        if valid and all_match:
            exact_match += 1

        j = "✓" if valid else "✗"
        e = "✓" if (valid and all_match) else "✗"
        cat_ok = "✓" if per_field.get("issue_category") else "✗"
        urg_ok = "✓" if per_field.get("urgency") else "✗"
        print(f"[{i:2d}]  {j}  {e}  {cat_ok:>3}  {urg_ok:>3}  {email[:45]}...")

        results.append({
            "i": i,
            "email": email,
            "expected": expected,
            "raw_output": raw,
            "predicted": predicted,
            "valid_json": valid,
            "exact_match": valid and all_match,
            "field_match": per_field,
        })

    n = len(examples)
    metrics = {
        "model": label,
        "n_examples": n,
        "json_valid_rate":  round(json_valid  / n, 4),
        "exact_match_rate": round(exact_match / n, 4),
        "field_accuracy":   {f: round(field_correct[f] / n, 4) for f in FIELDS},
        "details": results,
    }
    return metrics

print("✅ evaluate_model() function defined")

✅ evaluate_model() function defined


---
## 🧪 Step 2: Baseline — Llama 3.1 8B (no fine-tuning)

In [9]:
import gc
import mlx.core as mx
from mlx_lm import load

print(f"Loading {MODEL_ID}...")
print("⏱️  First run downloads ~4.5 GB (then cached in ~/.cache/huggingface/)")
print("   Subsequent runs load instantly from cache.\n")

model, tokenizer = load(MODEL_ID)

print("\n✅ Model loaded!")
print("   This is the plain 4-bit quantized model — no task-specific training yet.")

Loading mlx-community/Meta-Llama-3.1-8B-Instruct-4bit...
⏱️  First run downloads ~4.5 GB (then cached in ~/.cache/huggingface/)
   Subsequent runs load instantly from cache.



Fetching 6 files: 100%|██████████| 6/6 [07:03<00:00, 70.61s/it] 



✅ Model loaded!
   This is the plain 4-bit quantized model — no task-specific training yet.


In [10]:
# Quick sanity check — make sure the model can generate something reasonable
from mlx_lm import generate

test_email = "Hi, my Pro Plan was charged twice this month. Please refund. — Alex"
test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": test_email},
]
test_prompt = tokenizer.apply_chat_template(
    test_messages, add_generation_prompt=True, tokenize=False
)

print("Test email:", test_email)
print("\nModel raw output:")
test_output = generate(model, tokenizer, prompt=test_prompt, max_tokens=200, verbose=False)
print(test_output)

parsed, valid = safe_json_parse(test_output)
print(f"\nParsed JSON: {json.dumps(parsed, indent=2)}")
print(f"Valid JSON: {valid}")

Test email: Hi, my Pro Plan was charged twice this month. Please refund. — Alex

Model raw output:
{
  "customer_name": "Alex",
  "product": "Pro Plan",
  "issue_category": "billing",
  "urgency": "high",
  "summary": "Duplicate charge for Pro Plan this month"
}

Parsed JSON: {
  "customer_name": "Alex",
  "product": "Pro Plan",
  "issue_category": "billing",
  "urgency": "high",
  "summary": "Duplicate charge for Pro Plan this month"
}
Valid JSON: True


In [11]:
# Run full baseline evaluation on all 30 examples
# J = json valid, E = exact match, cat = issue_category correct, urg = urgency correct
print("Running baseline evaluation on 30 examples...")
print("This takes ~5-10 minutes on M5 16 GB.\n")

baseline_metrics = evaluate_model(
    model, tokenizer, eval_examples,
    label=f"{MODEL_ID} (base — no fine-tuning)"
)

Running baseline evaluation on 30 examples...
This takes ~5-10 minutes on M5 16 GB.


Evaluating: mlx-community/Meta-Llama-3.1-8B-Instruct-4bit (base — no fine-tuning)
  #  J  E  cat  urg  Preview
-----------------------------------------------------------------
[ 1]  ✓  ✗    ✗    ✓  Hi, can you tell me when the next maintenance...
[ 2]  ✓  ✗    ✓    ✓  Hello, two things: (1) my Pro Plan was double...
[ 3]  ✓  ✓    ✓    ✓  Production is down. API Access returns 500 on...
[ 4]  ✓  ✓    ✓    ✓  Oh great, my Premium Plan got cancelled AGAIN...
[ 5]  ✓  ✗    ✓    ✗  Hi, your software keeps freezing on my laptop...
[ 6]  ✓  ✗    ✓    ✗  Привіт! I cannot login to Mobile App. Passwor...
[ 7]  ✓  ✗    ✗    ✗  Why is my Pro Plan slower than my friend's Pr...
[ 8]  ✓  ✗    ✓    ✗  Cancel my account please. Premium Plan....
[ 9]  ✓  ✗    ✓    ✓  Love Analytics Add-on! Quick question — can i...
[10]  ✓  ✓    ✓    ✓  URGENT URGENT URGENT my Enterprise Plan accou...
[11]  ✓  ✗    ✓    ✓  Hello suppo

In [12]:
# Display baseline results with a nice summary
print("\n" + "="*65)
print("BASELINE RESULTS — Llama 3.1 8B (no fine-tuning)")
print("="*65)
print(f"JSON Valid:    {baseline_metrics['json_valid_rate']*100:5.1f}%")
print(f"Exact Match:   {baseline_metrics['exact_match_rate']*100:5.1f}%  (all 5 fields correct)")
print("\nField Accuracy:")
for f in FIELDS:
    acc = baseline_metrics['field_accuracy'][f] * 100
    bar = "█" * int(acc / 5) + "░" * (20 - int(acc / 5))
    print(f"  {f:<20} {acc:5.1f}%  [{bar}]")

print("\n" + "-"*65)
print("Baseline context (already run on other models):")
print("  Llama 3.3 70B  — Exact Match: 36.7%  |  Urgency: 63.3%")
print("  gpt-4o-mini    — Exact Match: 20.0%  |  Urgency: 43.3%")
print("-"*65)
print("\n📝 Write down these numbers — they're your 'before' comparison.")
print("   After fine-tuning, we should see improvement, especially on urgency.")


BASELINE RESULTS — Llama 3.1 8B (no fine-tuning)
JSON Valid:     93.3%
Exact Match:    43.3%  (all 5 fields correct)

Field Accuracy:
  customer_name         93.3%  [██████████████████░░]
  product               83.3%  [████████████████░░░░]
  issue_category        80.0%  [████████████████░░░░]
  urgency               63.3%  [████████████░░░░░░░░]
  summary               66.7%  [█████████████░░░░░░░]

-----------------------------------------------------------------
Baseline context (already run on other models):
  Llama 3.3 70B  — Exact Match: 36.7%  |  Urgency: 63.3%
  gpt-4o-mini    — Exact Match: 20.0%  |  Urgency: 43.3%
-----------------------------------------------------------------

📝 Write down these numbers — they're your 'before' comparison.
   After fine-tuning, we should see improvement, especially on urgency.


---
## 🗃️ Step 3: Prepare training data for MLX-LM

Split 300 examples into:
- **270 for training** — the model sees these and learns from them
- **30 for validation** — used to monitor for overfitting during training (loss goes up = overfitting)

**Important:** this validation split is different from `eval.jsonl` — those 30 never get touched during training.

In [13]:
import random
import shutil

# Load all 300 training examples
train_all = []
with open(TRAIN_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            train_all.append(json.loads(line))

print(f"Total training examples: {len(train_all)}")

# Shuffle and split
random.seed(42)
random.shuffle(train_all)
n_valid = 30
train_split = train_all[n_valid:]   # 270 for training
valid_split = train_all[:n_valid]   # 30 for validation (overfitting monitor)

print(f"Train split: {len(train_split)} examples")
print(f"Valid split: {len(valid_split)} examples")
print(f"Eval set   : {len(eval_examples)} examples (never touched during training!)"
      )

# Write to mlx_data/ directory
train_out = MLX_DATA_DIR / "train.jsonl"
valid_out  = MLX_DATA_DIR / "valid.jsonl"

with open(train_out, "w") as f:
    for ex in train_split:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

with open(valid_out, "w") as f:
    for ex in valid_split:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"\n✅ Written: {train_out}")
print(f"✅ Written: {valid_out}")

# Show one training example
print("\nSample training example (what the model trains on):")
sample = train_split[0]
for msg in sample["messages"]:
    role = msg["role"].upper()
    content_preview = msg["content"][:100]
    print(f"  [{role}]: {content_preview}...")

Total training examples: 300
Train split: 270 examples
Valid split: 30 examples
Eval set   : 30 examples (never touched during training!)

✅ Written: /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/lesson 17 - llm-fine-tuning-in-production/homework/mlx_data/train.jsonl
✅ Written: /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/lesson 17 - llm-fine-tuning-in-production/homework/mlx_data/valid.jsonl

Sample training example (what the model trains on):
  [SYSTEM]: You extract structured data from customer support emails. Return only a single valid JSON object wit...
  [USER]: Do you have a referral program for Premium Plan? Interested. — Lisa Anderson...
  [ASSISTANT]: {"customer_name": "Lisa Anderson", "product": "Premium Plan", "issue_category": "other", "urgency": ...


---
## 🚀 Step 4: Fine-Tune with LoRA


### Key hyperparameters (from homework spec)

| Parameter | Value | What it means |
|-----------|-------|---------------|
| `rank` (r) | 16 | LoRA matrix size — how much "capacity" the adapter has. Higher = more expressive but more params to train |
| `lora_layers` | 8 | How many transformer layers get LoRA adapters (out of 32 total). Less = faster training, less memory |
| `batch_size` | 2 | Examples processed per gradient step. Reduce to 1 if you get OOM errors |
| `iters` | 405 | Total gradient steps ≈ 3 epochs (270 examples ÷ batch 2 = 135 steps/epoch × 3) |
| `learning_rate` | 2e-5 | How big each weight update is. Too high = unstable training; too low = slow learning |
| `--mask-prompt` | on | Only train on the JSON output, NOT on the email/system prompt (this is label masking) |

In [14]:
# Free memory before training — the model uses the same unified memory pool
print("Releasing model from memory before training...")
del model, tokenizer
gc.collect()
mx.clear_cache()
print("✅ Memory released")
print("   (MLX-LM will reload the model internally during training)")

Releasing model from memory before training...
✅ Memory released
   (MLX-LM will reload the model internally during training)


mx.metal.clear_cache is deprecated and will be removed in a future version. Use mx.clear_cache instead.


In [18]:
import subprocess
import sys
import yaml

# ── Hyperparameters ──────────────────────────────────────────────
RANK        = 16   # LoRA rank
LORA_LAYERS = 8    # transformer layers to apply LoRA to (out of 32)
BATCH_SIZE  = 2    # reduce to 1 if OOM
ITERS       = 405  # ≈ 3 epochs (270 / 2 = 135 steps/epoch × 3)
LR          = 2e-5
SAVE_EVERY  = 100
# ─────────────────────────────────────────────────────────────────

# mlx-lm 0.21+ breaking changes:
#   --rank removed  → set via lora_parameters in a YAML config
#   --lora-layers   → renamed to --num-layers
#   python -m mlx_lm.lora → deprecated; use python -m mlx_lm lora

config = {
    "model":         MODEL_ID,
    "train":         True,
    "data":          str(MLX_DATA_DIR),
    "num_layers":    LORA_LAYERS,
    "batch_size":    BATCH_SIZE,
    "iters":         ITERS,
    "learning_rate": LR,
    "adapter_path":  str(ADAPTERS_DIR),
    "save_every":    SAVE_EVERY,
    "mask_prompt":   True,
    "lora_parameters": {
        "rank":    RANK,
        "dropout": 0.0,
        "scale":   20.0,
    },
}

config_path = HOMEWORK_DIR / "mlx_train_config.yaml"
with open(config_path, "w") as f:
    yaml.dump(config, f)

cmd = [sys.executable, "-m", "mlx_lm", "lora", "--config", str(config_path)]
print(f"Config: {config_path}")
print(f"Command: {' '.join(cmd)}")
print()
print("="*65)
print(f"Fine-tuning: {ITERS} steps (~3 epochs), rank={RANK}, layers={LORA_LAYERS}")
print("Expected time: 20-45 min on M5 16 GB")
print("Loss should drop from ~2.5 → ~0.5. Val loss rising = overfitting.")
print("OOM? → set BATCH_SIZE=1 and LORA_LAYERS=4 above, then re-run.")
print("="*65 + "\n")

# Popen streams output line-by-line into the cell during training
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()

if proc.returncode == 0:
    print("\n✅ Fine-tuning complete!")
    for f in sorted(ADAPTERS_DIR.glob("*")):
        print(f"  {f.name:<35} {f.stat().st_size/1024/1024:.1f} MB")
else:
    print(f"\n❌ Training failed (exit {proc.returncode})")
    print("  OOM        → BATCH_SIZE=1, LORA_LAYERS=4")
    print("  yaml error → pip install pyyaml")
    print("  auth error → re-run HuggingFace login cell")

Config: /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/lesson 17 - llm-fine-tuning-in-production/homework/mlx_train_config.yaml
Command: /opt/homebrew/opt/python@3.14/bin/python3.14 -m mlx_lm lora --config /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/lesson 17 - llm-fine-tuning-in-production/homework/mlx_train_config.yaml

Fine-tuning: 405 steps (~3 epochs), rank=16, layers=8
Expected time: 20-45 min on M5 16 GB
Loss should drop from ~2.5 → ~0.5. Val loss rising = overfitting.
OOM? → set BATCH_SIZE=1 and LORA_LAYERS=4 above, then re-run.

Loading configuration file /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/lesson 17 - llm-fine-tuning-in-production/homework/mlx_train_config.yaml
Loading pretrained model

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 5807.94it/s]
Loading datasets
Training
Trainable parameters: 0.131% (10.486M/8030.261M)
Starting training..., iters: 405

Calculating loss...: 100%|██████████| 15/15 [00:10<00:00,

---
## 📈 Step 5: Evaluate the fine-tuned model

In [19]:
# Free any cached memory first
gc.collect()
mx.metal.clear_cache()

# Load base model + LoRA adapter
print(f"Loading fine-tuned model: {MODEL_ID}")
print(f"With adapter from: {ADAPTERS_DIR}")
print("(Loads quickly from cache this time)\n")

model_ft, tokenizer_ft = load(MODEL_ID, adapter_path=str(ADAPTERS_DIR))

print("\n✅ Fine-tuned model loaded!")
print("   = base model weights (frozen) + LoRA adapter (trained)")

Loading fine-tuned model: mlx-community/Meta-Llama-3.1-8B-Instruct-4bit
With adapter from: /Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/lesson 17 - llm-fine-tuning-in-production/homework/adapters
(Loads quickly from cache this time)



Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 122760.12it/s]



✅ Fine-tuned model loaded!
   = base model weights (frozen) + LoRA adapter (trained)


In [20]:
# Quick test with the same email we used for the baseline
from mlx_lm import generate

test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": test_email},
]
test_prompt_ft = tokenizer_ft.apply_chat_template(
    test_messages, add_generation_prompt=True, tokenize=False
)

print("Test email:", test_email)
print("\nFine-tuned model output:")
test_output_ft = generate(model_ft, tokenizer_ft, prompt=test_prompt_ft, max_tokens=200, verbose=False)
print(test_output_ft)

parsed_ft, valid_ft = safe_json_parse(test_output_ft)
print(f"\nParsed: {json.dumps(parsed_ft, indent=2)}")
print(f"Valid:  {valid_ft}")

Test email: Hi, my Pro Plan was charged twice this month. Please refund. — Alex

Fine-tuned model output:
{"customer_name": "Alex", "product": "Pro Plan", "issue_category": "billing", "urgency": "high", "summary": "Duplicate charge for Pro Plan, refund requested"}

Parsed: {
  "customer_name": "Alex",
  "product": "Pro Plan",
  "issue_category": "billing",
  "urgency": "high",
  "summary": "Duplicate charge for Pro Plan, refund requested"
}
Valid:  True


In [21]:
# Run full evaluation on all 30 examples (same as baseline)
print("Running fine-tuned evaluation on 30 examples...\n")

finetuned_metrics = evaluate_model(
    model_ft, tokenizer_ft, eval_examples,
    label=f"{MODEL_ID} (fine-tuned, r={RANK}, layers={LORA_LAYERS})"
)

Running fine-tuned evaluation on 30 examples...


Evaluating: mlx-community/Meta-Llama-3.1-8B-Instruct-4bit (fine-tuned, r=16, layers=8)
  #  J  E  cat  urg  Preview
-----------------------------------------------------------------
[ 1]  ✓  ✓    ✓    ✓  Hi, can you tell me when the next maintenance...
[ 2]  ✓  ✗    ✓    ✓  Hello, two things: (1) my Pro Plan was double...
[ 3]  ✓  ✗    ✓    ✗  Production is down. API Access returns 500 on...
[ 4]  ✓  ✓    ✓    ✓  Oh great, my Premium Plan got cancelled AGAIN...
[ 5]  ✓  ✗    ✓    ✗  Hi, your software keeps freezing on my laptop...
[ 6]  ✓  ✗    ✓    ✓  Привіт! I cannot login to Mobile App. Passwor...
[ 7]  ✓  ✗    ✗    ✗  Why is my Pro Plan slower than my friend's Pr...
[ 8]  ✓  ✓    ✓    ✓  Cancel my account please. Premium Plan....
[ 9]  ✓  ✗    ✓    ✓  Love Analytics Add-on! Quick question — can i...
[10]  ✓  ✗    ✓    ✗  URGENT URGENT URGENT my Enterprise Plan accou...
[11]  ✓  ✓    ✓    ✓  Hello support team, I cancelled my Premium

---
## 📊 Step 6: Compare results

In [22]:
def print_comparison_table(base, ft):
    """Print a nicely formatted comparison table."""
    print("\n" + "="*75)
    print(f"{'METRIC':<28} {'BASE 8B':>10} {'FT 8B':>10} {'LIFT':>10} {'NOTE'}")
    print("="*75)

    rows = [
        ("JSON Valid",        "json_valid_rate",  None,              "should be 100%"),
        ("Exact Match (all)", "exact_match_rate", None,              "all 5 fields correct"),
        ("─ customer_name",   "f/customer_name",  "customer_name",   ""),
        ("─ product",         "f/product",        "product",         ""),
        ("─ issue_category",  "f/issue_category", "issue_category",  "← routing"),
        ("─ urgency",         "f/urgency",        "urgency",         "← escalation ⚠️"),
        ("─ summary",         "f/summary",        "summary",         ""),
    ]

    for label, key, field, note in rows:
        if key.startswith("f/"):
            b_val  = base["field_accuracy"][field]
            ft_val = ft["field_accuracy"][field]
        else:
            b_val  = base[key]
            ft_val = ft[key]

        lift = ft_val - b_val
        lift_str = f"+{lift*100:.1f}%" if lift > 0 else (f"{lift*100:.1f}%" if lift < 0 else " 0.0%")
        arrow = "▲" if lift > 0.01 else ("▼" if lift < -0.01 else "—")

        print(f"{label:<28} {b_val*100:>9.1f}%  {ft_val*100:>8.1f}%  {arrow} {lift_str:>6}  {note}")

    print("="*75)
    print()
    print("For reference — previously evaluated models:")
    print(f"  {'Llama 3.3 70B (Together API)':<28} {'—':>10} {'36.7%':>10} {'—':>10}  urgency: 63.3%")
    print(f"  {'gpt-4o-mini'::<28} {'—':>10} {'20.0%':>10} {'—':>10}  urgency: 43.3%")
    print()
    print("Key question: Did fine-tuned 8B beat 70B on urgency accuracy?")


print_comparison_table(baseline_metrics, finetuned_metrics)


METRIC                          BASE 8B      FT 8B       LIFT NOTE
JSON Valid                        93.3%     100.0%  ▲  +6.7%  should be 100%
Exact Match (all)                 43.3%      73.3%  ▲ +30.0%  all 5 fields correct
─ customer_name                   93.3%     100.0%  ▲  +6.7%  
─ product                         83.3%      90.0%  ▲  +6.7%  
─ issue_category                  80.0%      96.7%  ▲ +16.7%  ← routing
─ urgency                         63.3%      86.7%  ▲ +23.3%  ← escalation ⚠️
─ summary                         66.7%      80.0%  ▲ +13.3%  

For reference — previously evaluated models:
  Llama 3.3 70B (Together API)          —      36.7%          —  urgency: 63.3%
  gpt-4o-mini:::::::::::::::::          —      20.0%          —  urgency: 43.3%

Key question: Did fine-tuned 8B beat 70B on urgency accuracy?


In [23]:
# Look at specific examples where urgency improved or got worse
print("\nDetailed urgency analysis — where did we improve?\n")
print(f"{'#':>3}  {'Expected':>10}  {'Base':>12}  {'FT':>12}  {'Improved?'}")
print("-"*60)

base_details = {d['i']: d for d in baseline_metrics['details']}
ft_details   = {d['i']: d for d in finetuned_metrics['details']}

improvements = 0
regressions  = 0

for i in range(1, 31):
    b  = base_details[i]
    ft = ft_details[i]

    exp_urg  = b['expected']['urgency']
    b_urg    = b['predicted']['urgency']  if (b['predicted']  and b['valid_json'])  else "(invalid)"
    ft_urg   = ft['predicted']['urgency'] if (ft['predicted'] and ft['valid_json']) else "(invalid)"

    b_ok  = b['field_match'].get('urgency', False)
    ft_ok = ft['field_match'].get('urgency', False)

    if b_ok == ft_ok:
        status = "same"
    elif not b_ok and ft_ok:
        status = "✅ improved"
        improvements += 1
    else:
        status = "❌ regressed"
        regressions += 1

    if status != "same":  # Only show changes
        print(f"[{i:2d}]  {exp_urg:>10}  {str(b_urg):>12}  {str(ft_urg):>12}  {status}")

print("-"*60)
print(f"Urgency improvements: {improvements}")
print(f"Urgency regressions:  {regressions}")


Detailed urgency analysis — where did we improve?

  #    Expected          Base            FT  Improved?
------------------------------------------------------------
[ 3]    critical      critical        medium  ❌ regressed
[ 6]      medium          high        medium  ✅ improved
[ 8]      medium           low        medium  ✅ improved
[10]    critical      critical          high  ❌ regressed
[12]      medium           low        medium  ✅ improved
[15]      medium          high        medium  ✅ improved
[16]      medium          high        medium  ✅ improved
[18]      medium          high        medium  ✅ improved
[19]      medium          high        medium  ✅ improved
[20]      medium     (invalid)        medium  ✅ improved
[26]    critical     (invalid)      critical  ✅ improved
------------------------------------------------------------
Urgency improvements: 9
Urgency regressions:  2


---
## 💾 Step 7: Save all results

In [24]:
import datetime

# Save baseline results
baseline_out   = RESULTS_DIR / "baseline_8b_mlx.json"
finetuned_out  = RESULTS_DIR / "finetuned_8b_mlx.json"
comparison_out = RESULTS_DIR / "comparison_summary.json"

with open(baseline_out, "w") as f:
    json.dump(baseline_metrics, f, indent=2, ensure_ascii=False)

with open(finetuned_out, "w") as f:
    json.dump(finetuned_metrics, f, indent=2, ensure_ascii=False)

# Compact comparison summary (no per-example details — easier to read in README)
comparison = {
    "generated_at": datetime.datetime.now().isoformat(),
    "model_id": MODEL_ID,
    "hyperparameters": {
        "rank": RANK, "lora_layers": LORA_LAYERS,
        "batch_size": BATCH_SIZE, "iters": ITERS, "learning_rate": LR
    },
    "eval_set_size": len(eval_examples),
    "base": {
        "json_valid_rate":  baseline_metrics["json_valid_rate"],
        "exact_match_rate": baseline_metrics["exact_match_rate"],
        "field_accuracy":   baseline_metrics["field_accuracy"],
    },
    "finetuned": {
        "json_valid_rate":  finetuned_metrics["json_valid_rate"],
        "exact_match_rate": finetuned_metrics["exact_match_rate"],
        "field_accuracy":   finetuned_metrics["field_accuracy"],
    },
    "lift": {
        "exact_match": round(finetuned_metrics["exact_match_rate"] - baseline_metrics["exact_match_rate"], 4),
        "urgency":     round(finetuned_metrics["field_accuracy"]["urgency"] - baseline_metrics["field_accuracy"]["urgency"], 4),
    },
    "reference_models": {
        "llama_3.3_70b": {"exact_match_rate": 0.3667, "urgency": 0.6333},
        "gpt_4o_mini":   {"exact_match_rate": 0.2000, "urgency": 0.4333},
    }
}

with open(comparison_out, "w") as f:
    json.dump(comparison, f, indent=2, ensure_ascii=False)

print("Results saved:")
for path in [baseline_out, finetuned_out, comparison_out]:
    size_kb = path.stat().st_size / 1024
    print(f"  ✅ {path.name:<35} {size_kb:.1f} KB")

print(f"\nAdapter weights:")
for f in sorted(ADAPTERS_DIR.glob("*")):
    size_mb = f.stat().st_size / (1024*1024)
    print(f"  ✅ {f.name:<35} {size_mb:.1f} MB")

Results saved:
  ✅ baseline_8b_mlx.json                32.6 KB
  ✅ finetuned_8b_mlx.json               30.4 KB
  ✅ comparison_summary.json             1.0 KB

Adapter weights:
  ✅ 0000100_adapters.safetensors        40.0 MB
  ✅ 0000200_adapters.safetensors        40.0 MB
  ✅ 0000300_adapters.safetensors        40.0 MB
  ✅ 0000400_adapters.safetensors        40.0 MB
  ✅ adapter_config.json                 0.0 MB
  ✅ adapters.safetensors                40.0 MB


---
## 📝 Step 8: README report numbers

In [28]:
# Print everything you need for the README report

b = baseline_metrics
ft = finetuned_metrics

print("="*70)
print("README REPORT — copy these numbers")
print("="*70)

print("""
### 1. Comparison Table

| Metric              | Base 8B  | Fine-Tuned 8B    | Llama 3.3 70B   |
|---------------------|----------|------------------|-----------------|"""
)
rows = [
    ("JSON Valid",       b["json_valid_rate"],          ft["json_valid_rate"],          1.0),
    ("Exact Match",      b["exact_match_rate"],         ft["exact_match_rate"],         0.367),
    ("customer_name",    b["field_accuracy"]["customer_name"],  ft["field_accuracy"]["customer_name"],  0.967),
    ("product",          b["field_accuracy"]["product"],        ft["field_accuracy"]["product"],        0.900),
    ("issue_category",   b["field_accuracy"]["issue_category"], ft["field_accuracy"]["issue_category"], 0.833),
    ("urgency",          b["field_accuracy"]["urgency"],        ft["field_accuracy"]["urgency"],        0.633),
    ("summary",          b["field_accuracy"]["summary"],        ft["field_accuracy"]["summary"],        0.667),
]
for label, b_val, ft_val, ref_val in rows:
    lift = ft_val - b_val
    lift_str = f"(+{lift*100:.0f}pp)" if lift > 0 else (f"({lift*100:.0f}pp)" if lift < 0 else "")
    print(f"| {label:<19} | {b_val*100:6.1f}%  | {ft_val*100:6.1f}% {lift_str:<8} | {ref_val*100:6.1f}%         |")

print("""\n### 2. Cost & Breakeven

- Training cost: $0 (local compute on Apple Silicon)
- Model size: ~4.5 GB (4-bit), adapter: ~50 MB
- Inference: self-hosted on M5 vs API costs
  - Together AI Llama 3.3 70B: ~$0.88/M input tokens + $0.88/M output tokens
  - At 50K emails/day × ~130 input tokens × $0.88/M ≈ $5.72/day ≈ $172/month
  - Self-hosted 8B: electricity cost only (~$0-5/month)
  - Break-even vs API: immediately profitable if hardware is already owned
""")

print("Hyp params used:")
print(f"  rank={RANK}, lora_layers={LORA_LAYERS}, batch={BATCH_SIZE}, iters={ITERS}, lr={LR}")

README REPORT — copy these numbers

### 1. Comparison Table

| Metric              | Base 8B  | Fine-Tuned 8B    | Llama 3.3 70B   |
|---------------------|----------|------------------|-----------------|
| JSON Valid          |   93.3%  |  100.0% (+7pp)   |  100.0%         |
| Exact Match         |   43.3%  |   73.3% (+30pp)  |   36.7%         |
| customer_name       |   93.3%  |  100.0% (+7pp)   |   96.7%         |
| product             |   83.3%  |   90.0% (+7pp)   |   90.0%         |
| issue_category      |   80.0%  |   96.7% (+17pp)  |   83.3%         |
| urgency             |   63.3%  |   86.7% (+23pp)  |   63.3%         |
| summary             |   66.7%  |   80.0% (+13pp)  |   66.7%         |

### 2. Cost & Breakeven

- Training cost: $0 (local compute on Apple Silicon)
- Model size: ~4.5 GB (4-bit), adapter: ~50 MB
- Inference: self-hosted on M5 vs API costs
  - Together AI Llama 3.3 70B: ~$0.88/M input tokens + $0.88/M output tokens
  - At 50K emails/day × ~130 input tokens × 

---
## 🔍 Analysis: Inspect failure cases

Understanding *where* the model still fails is important for the "What didn't work" section of the README.

In [29]:
# Show all examples where fine-tuned model still gets urgency wrong
print("Cases where fine-tuned model still fails on urgency:\n")

for d in finetuned_metrics['details']:
    if not d['field_match'].get('urgency', True):
        exp  = d['expected']['urgency']
        pred = d['predicted']['urgency'] if d['predicted'] else '(no JSON)'
        print(f"[{d['i']:2d}] Expected: {exp:<10}  Predicted: {pred}")
        print(f"     Email: {d['email'][:80]}...")
        print()

print("\nThese are the edge cases / ambiguous emails. Patterns to note:")
print("  - Sarcasm makes urgency hard to detect")
print("  - Multi-issue emails (billing + feature request) mix urgency signals")
print("  - Anonymous emails lack context clues about urgency")
print("  - Only 300 training examples may not cover all edge case patterns")

Cases where fine-tuned model still fails on urgency:

[ 3] Expected: critical    Predicted: medium
     Email: Production is down. API Access returns 500 on every call. Customers complaining....

[ 5] Expected: medium      Predicted: high
     Email: Hi, your software keeps freezing on my laptop. Worked fine before update. — Davi...

[ 7] Expected: low         Predicted: medium
     Email: Why is my Pro Plan slower than my friend's Pro Plan? Are different tiers throttl...

[10] Expected: critical    Predicted: high
     Email: URGENT URGENT URGENT my Enterprise Plan account locked, send help, all caps beca...


These are the edge cases / ambiguous emails. Patterns to note:
  - Sarcasm makes urgency hard to detect
  - Multi-issue emails (billing + feature request) mix urgency signals
  - Anonymous emails lack context clues about urgency
  - Only 300 training examples may not cover all edge case patterns


In [30]:
# Summary of failures across all fields in fine-tuned model
print("All remaining failures in fine-tuned model:\n")

for d in finetuned_metrics['details']:
    failed_fields = [f for f in FIELDS if not d['field_match'].get(f, True)]
    if failed_fields:
        print(f"[{d['i']:2d}] Failed: {', '.join(failed_fields)}")
        print(f"     Email: {d['email'][:70]}...")
        for field in failed_fields:
            exp_val  = d['expected'].get(field)
            pred_val = d['predicted'].get(field) if d['predicted'] else None
            print(f"     {field}: expected={exp_val!r}  predicted={pred_val!r}")
        print()

All remaining failures in fine-tuned model:

[ 2] Failed: summary
     Email: Hello, two things: (1) my Pro Plan was double charged this month, (2) ...
     summary: expected='Duplicate Pro Plan charge with secondary dark mode feature request'  predicted='Double charge for Pro Plan, refund is priority'

[ 3] Failed: urgency, summary
     Email: Production is down. API Access returns 500 on every call. Customers co...
     urgency: expected='critical'  predicted='medium'
     summary: expected='Production down, API returning 500'  predicted='API Access returning 500'

[ 5] Failed: product, urgency, summary
     Email: Hi, your software keeps freezing on my laptop. Worked fine before upda...
     product: expected='Desktop App'  predicted='software'
     urgency: expected='medium'  predicted='high'
     summary: expected='Desktop app freezing after update'  predicted='Software freezes on laptop'

[ 6] Failed: summary
     Email: Привіт! I cannot login to Mobile App. Password is correct. 

---
## ✅ Data integrity: overlap check 

## Results are too good to be true. Let's check overlap.

In [31]:
import hashlib
import json
from pathlib import Path

def email_hash(text: str) -> str:
    """Normalize and hash an email string for dedup comparison."""
    normalized = " ".join(text.lower().split())
    return hashlib.sha256(normalized.encode()).hexdigest()

def load_emails(path: Path) -> dict[str, str]:
    """Load a JSONL file and return {hash: email} mapping."""
    emails = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            # eval.jsonl  → {"email": "...", "expected": {...}}
            # train.jsonl → {"messages": [{"role": "user", "content": "..."}, ...]}
            if "email" in obj:
                text = obj["email"]
            else:
                user_msgs = [m["content"] for m in obj.get("messages", []) if m["role"] == "user"]
                text = user_msgs[0] if user_msgs else ""
            h = email_hash(text)
            emails[h] = text
    return emails

# Load all three splits
eval_emails  = load_emails(EVAL_FILE)
train_emails = load_emails(MLX_DATA_DIR / "train.jsonl")
val_emails   = load_emails(MLX_DATA_DIR / "valid.jsonl")

print(f"eval  : {len(eval_emails):>4} examples")
print(f"train : {len(train_emails):>4} examples")
print(f"val   : {len(val_emails):>4} examples")
print(f"total : {len(eval_emails) + len(train_emails) + len(val_emails):>4} examples")
print()

# Check every pair of splits for overlap
checks = [
    ("eval  ∩ train", eval_emails,  train_emails),
    ("eval  ∩ val  ", eval_emails,  val_emails),
    ("train ∩ val  ", train_emails, val_emails),
]

all_clean = True
for label, a, b in checks:
    overlap = set(a.keys()) & set(b.keys())
    if overlap:
        all_clean = False
        print(f"❌ {label}: {len(overlap)} duplicate(s) found!")
        for h in list(overlap)[:3]:
            print(f"   {a[h][:80]}...")
    else:
        print(f"✅ {label}: 0 overlaps")

print()
if all_clean:
    print("✅ All splits are clean — no data leakage.")
else:
    print("❌ Overlap detected — eval results may be inflated. Re-generate data.")

eval  :   30 examples
train :  270 examples
val   :   30 examples
total :  330 examples

✅ eval  ∩ train: 0 overlaps
✅ eval  ∩ val  : 0 overlaps
✅ train ∩ val  : 0 overlaps

✅ All splits are clean — no data leakage.
